In [45]:
import pandas as pd
import os
import plotly.graph_objects as go

In [46]:
input_folder = "./input/"

# Only import column 2215, 2206, 2257
data = pd.read_csv(os.path.join(input_folder, "peaks.csv"))

In [47]:
# Convert FECHA_SARANDI, FECHA_POLANCO and FECHA_DURAZNO to datetime. recall NaT means Not a Time, which is used for missing values in datetime columns. Also convert to format dd-MM-yyyy hh:mm:s
data['FECHA_SARANDI'] = pd.to_datetime(data['FECHA_SARANDI'], errors='coerce').dt.strftime('%d-%m-%Y %H:%M:%S')
data['FECHA_POLANCO'] = pd.to_datetime(data['FECHA_POLANCO'], errors='coerce').dt.strftime('%d-%m-%Y %H:%M:%S')
data['FECHA_DURAZNO'] = pd.to_datetime(data['FECHA_DURAZNO'], errors='coerce').dt.strftime('%d-%m-%Y %H:%M:%S')

In [48]:
# set ID to index
data.set_index('ID', inplace=True)
data

,FECHA_SARANDI,SARANDI,FECHA_POLANCO,POLANCO,FECHA_DURAZNO,DURAZNO
ID,,,,,,
1,03-11-2009 23:00:00,5.62,05-11-2009 21:00:00,8.54,07-11-2009 07:00:00,7.94
2,20-11-2009 03:00:00,6.62,22-11-2009 02:00:00,9.85,23-11-2009 17:00:00,9.26
3,27-11-2009 18:00:00,4.24,28-11-2009 21:00:00,6.75,29-11-2009 13:00:00,6.64
4,05-02-2010 03:00:00,5.88,08-02-2010 11:00:00,12.05,09-02-2010 14:00:00,12.46
5,NaN,NaN,30-06-2010 10:00:00,7.23,01-07-2010 17:00:00,6.06
...,...,...,...,...,...,...
93,NaN,NaN,17-04-2026 00:00:00,6.40,18-04-2026 09:00:00,7.70
94,09-05-2026 00:00:00,5.10,11-05-2026 00:00:00,7.49,12-05-2026 08:00:00,7.25
95,20-07-2026 01:00:00,7.50,22-07-2026 06:00:00,9.85,24-07-2026 05:00:00,10.01


In [49]:
fmt = '%d-%m-%Y %H:%M:%S'
data['POLANCO_2_DURAZNO'] = (pd.to_datetime(data['FECHA_DURAZNO'], format=fmt, errors='coerce') - pd.to_datetime(data['FECHA_POLANCO'], format=fmt, errors='coerce')).dt.total_seconds() / 3600
data['SARANDI_2_DURAZNO'] = (pd.to_datetime(data['FECHA_DURAZNO'], format=fmt, errors='coerce') - pd.to_datetime(data['FECHA_SARANDI'], format=fmt, errors='coerce')).dt.total_seconds() / 3600
data['SARANDI_2_POLANCO'] = (pd.to_datetime(data['FECHA_POLANCO'], format=fmt, errors='coerce') - pd.to_datetime(data['FECHA_SARANDI'], format=fmt, errors='coerce')).dt.total_seconds() / 3600



In [50]:
data

,FECHA_SARANDI,SARANDI,FECHA_POLANCO,POLANCO,FECHA_DURAZNO,DURAZNO,POLANCO_2_DURAZNO,SARANDI_2_DURAZNO,SARANDI_2_POLANCO
ID,,,,,,,,,
1,03-11-2009 23:00:00,5.62,05-11-2009 21:00:00,8.54,07-11-2009 07:00:00,7.94,34.0,80.0,46.0
2,20-11-2009 03:00:00,6.62,22-11-2009 02:00:00,9.85,23-11-2009 17:00:00,9.26,39.0,86.0,47.0
3,27-11-2009 18:00:00,4.24,28-11-2009 21:00:00,6.75,29-11-2009 13:00:00,6.64,16.0,43.0,27.0
4,05-02-2010 03:00:00,5.88,08-02-2010 11:00:00,12.05,09-02-2010 14:00:00,12.46,27.0,107.0,80.0
5,NaN,NaN,30-06-2010 10:00:00,7.23,01-07-2010 17:00:00,6.06,31.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
93,NaN,NaN,17-04-2026 00:00:00,6.40,18-04-2026 09:00:00,7.70,33.0,NaN,NaN
94,09-05-2026 00:00:00,5.10,11-05-2026 00:00:00,7.49,12-05-2026 08:00:00,7.25,32.0,80.0,48.0
95,20-07-2026 01:00:00,7.50,22-07-2026 06:00:00,9.85,24-07-2026 05:00:00,10.01,47.0,100.0,53.0


In [ ]:
from plotly.subplots import make_subplots
import numpy as np
import ipywidgets as widgets
from IPython.display import display

site_labels = {
    'SARANDI': 'Sarandi del Yí',
    'POLANCO': 'Polanco del Yí',
    'DURAZNO': 'Durazno R5',
}

x_widget = widgets.Dropdown(options=[(label, code) for code, label in site_labels.items()], value='POLANCO', description='Eje X:')
y_widget = widgets.Dropdown(options=[(label, code) for code, label in site_labels.items()], value='DURAZNO', description='Eje Y:')
output = widgets.Output()

def get_transit_time(site_a, site_b):
    # tiempo de tránsito de site_a a site_b, invirtiendo el signo si solo existe la columna inversa
    col_ab = f'{site_a}_2_{site_b}'
    col_ba = f'{site_b}_2_{site_a}'
    if col_ab in data.columns:
        return data[col_ab]
    if col_ba in data.columns:
        return -data[col_ba]
    return None

def plot_sites(*_):
    x_site, y_site = x_widget.value, y_widget.value
    with output:
        output.clear_output(wait=True)
        if x_site == y_site:
            print('Selecciona dos sitios distintos.')
            return

        fig = make_subplots(rows=2, cols=1)
        row1_domain = fig.layout.yaxis.domain
        row2_domain = fig.layout.yaxis2.domain

        mask = data[x_site].notna() & data[y_site].notna()
        x = data[x_site][mask].values
        y = data[y_site][mask].values
        coeffs = np.polyfit(x, y, 1)
        trendline_x = np.linspace(x.min(), x.max(), 100)
        trendline_y = np.polyval(coeffs, trendline_x)

        residuals = y - np.polyval(coeffs, x)
        std = np.std(residuals)

        equation = f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f}'

        fig.add_trace(
            go.Scatter(x=data[x_site], y=data[y_site], mode='markers', name='Datos observados', legend='legend', showlegend=True),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=trendline_x, y=trendline_y, mode='lines', name=f'Tendencia lineal ({equation})', line=dict(color='red'), legend='legend', showlegend=True),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=np.concatenate([trendline_x, trendline_x[::-1]]),
                y=np.concatenate([trendline_y + std, (trendline_y - std)[::-1]]),
                fill='toself', fillcolor='rgba(255,0,0,0.2)', line=dict(color='rgba(255,255,255,0)'),
                name='Banda de incertidumbre (±1 std)', legend='legend', showlegend=True
            ),
            row=1, col=1
        )

        transit = get_transit_time(x_site, y_site)
        if transit is not None:
            fig.add_trace(
                go.Scatter(x=transit, y=data[y_site], mode='markers', name='Tiempo de tránsito', legend='legend2', showlegend=True),
                row=2, col=1
            )
            fig.update_xaxes(title_text=f'Tiempo tránsito {site_labels[x_site]} → {site_labels[y_site]} (horas)', row=2, col=1)
            fig.update_yaxes(title_text=site_labels[y_site], row=2, col=1)

        fig.add_annotation(
            x=trendline_x[10], y=trendline_y[10],
            text=equation,
            showarrow=True, arrowhead=2,
            font=dict(color='red'),
            xref='x', yref='y'
        )

        fig.update_xaxes(title_text=site_labels[x_site], dtick=1, minor=dict(dtick=0.5, showgrid=True), row=1, col=1)
        fig.update_yaxes(title_text=site_labels[y_site], dtick=1, minor=dict(dtick=0.5, showgrid=True), row=1, col=1)

        fig.update_layout(
            width=800, height=1300,
            legend=dict(orientation='h', x=0.5, xanchor='center', y=row1_domain[0] - 0.08, yanchor='top'),
            legend2=dict(orientation='h', x=0.5, xanchor='center', y=row2_domain[0] - 0.12, yanchor='top'),
            margin=dict(b=140),
        )
        fig.show()

x_widget.observe(plot_sites, names='value')
y_widget.observe(plot_sites, names='value')

display(widgets.HBox([x_widget, y_widget]), output)
plot_sites()


Output()

In [81]:
from sklearn.linear_model import LinearRegression

# Modelos individuales
# POLANCO -> DURAZNO
mask_p = data['POLANCO'].notna() & data['DURAZNO'].notna()
x_p = data.loc[mask_p, 'POLANCO'].values.reshape(-1, 1)
y_p = data.loc[mask_p, 'DURAZNO'].values
coeffs_p = np.polyfit(x_p.flatten(), y_p, 1)
residuals_p = y_p - np.polyval(coeffs_p, x_p.flatten())
std_p = np.std(residuals_p)

# SARANDI -> DURAZNO
mask_s = data['SARANDI'].notna() & data['DURAZNO'].notna()
x_s = data.loc[mask_s, 'SARANDI'].values.reshape(-1, 1)
y_s = data.loc[mask_s, 'DURAZNO'].values
coeffs_s = np.polyfit(x_s.flatten(), y_s, 1)
residuals_s = y_s - np.polyval(coeffs_s, x_s.flatten())
std_s = np.std(residuals_s)

def predecir_durazno(polanco_val=None, sarandi_val=None, n_std=1):
    """
    Predice el nivel máximo en Durazno según los valores disponibles.
    
    Parámetros:
        polanco_val: nivel en Polanco del Yí (opcional)
        sarandi_val: nivel en Sarandi del Yí (opcional)
        n_std: número de desviaciones estándar para el rango de incertidumbre
    
    Retorna:
        valor_predicho, (rango_min, rango_max), modelo_usado
    """
    if polanco_val is not None and sarandi_val is not None:
        # Modelo múltiple: POLANCO + SARANDI -> DURAZNO
        valor = a_polanco * polanco_val + b_sarandi * sarandi_val + c_intercept
        std_used = std_ps
        modelo = 'Polanco + Sarandi (regresión múltiple)'
    elif polanco_val is not None:
        # Modelo simple: POLANCO -> DURAZNO
        valor = np.polyval(coeffs_p, polanco_val)
        std_used = std_p
        modelo = 'Solo Polanco del Yí (regresión simple)'
    elif sarandi_val is not None:
        # Modelo simple: SARANDI -> DURAZNO
        valor = np.polyval(coeffs_s, sarandi_val)
        std_used = std_s
        modelo = 'Solo Sarandi del Yí (regresión simple)'
    else:
        raise ValueError("Debe proporcionar al menos un valor: polanco_val o sarandi_val.")
    
    return valor, (valor - n_std * std_used, valor + n_std * std_used), modelo






In [82]:
# Ejemplo de uso
print("--- Solo Polanco ---")
v, (rmin, rmax), modelo = predecir_durazno(polanco_val=10.31)
print(f"Modelo: {modelo}\nPredicción: {v:.2f}, Rango ±1std: [{rmin:.2f}, {rmax:.2f}]\n")



--- Solo Polanco ---
Modelo: Solo Polanco del Yí (regresión simple)
Predicción: 10.41, Rango ±1std: [9.83, 10.98]



In [85]:
print("--- Solo Sarandi ---")
v, (rmin, rmax), modelo = predecir_durazno(sarandi_val=7.16)
print(f"Modelo: {modelo}\nPredicción: {v:.2f}, Rango ±1std: [{rmin:.2f}, {rmax:.2f}]\n")



--- Solo Sarandi ---
Modelo: Solo Sarandi del Yí (regresión simple)
Predicción: 10.36, Rango ±1std: [8.97, 11.75]



In [84]:
print("--- Polanco + Sarandi ---")
v, (rmin, rmax), modelo = predecir_durazno(polanco_val=10.31, sarandi_val=7.16)
print(f"Modelo: {modelo}\nPredicción: {v:.2f}, Rango ±1std: [{rmin:.2f}, {rmax:.2f}]")

--- Polanco + Sarandi ---
Modelo: Polanco + Sarandi (regresión múltiple)
Predicción: 10.29, Rango ±1std: [9.83, 10.75]
